--------------------------
#### 오늘의 실습 목표 (타이타닉 버전)
- 결측치/범주형 처리(전처리)
- Pipeline으로 “전처리 + 모델” 한 번에 묶기
- train_test_split(test_size, random_state)가 결과에 미치는 영향 실험
- 모델 파라미터(예: RandomForest의 max_depth)가 과적합에 미치는 영향 실험
- cross_val_score로 성능을 안정적으로 보기
--------------------------

In [ ]:
#  타이타닉 데이터셋 불러오기
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

# TODO: 목표 변수(y)와 입력 변수(X)로 분리하세요.
# 예: y = df["Survived"], X = df.drop(columns=["Survived"])

In [ ]:
# 모델에 필요한 것들 불러오기
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

In [ ]:
# TODO: 숫자/문자 컬럼 자동 분리
# 힌트: X.select_dtypes로 int/float과 object를 나누세요.
num_cols = None
cat_cols = None

# TODO: 숫자 컬럼 전처리 파이프라인 작성
# 힌트: SimpleImputer(strategy="median")
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# TODO: 범주형 컬럼 전처리 파이프라인 작성
# 힌트: SimpleImputer(strategy="most_frequent") + OneHotEncoder(handle_unknown="ignore")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# TODO: ColumnTransformer로 전처리 묶기
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: 모델 생성 (로지스틱 회귀)
model = LogisticRegression(max_iter=1000)

# TODO: 전처리 + 모델 파이프라인 만들기
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# TODO: 학습/검증 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TODO: 학습 및 예측
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

# TODO: 정확도 출력
print("Accuracy:", accuracy_score(y_test, pred))

In [ ]:
# TODO: test_size를 바꿔가며 정확도 확인
for test_size in [0.1, 0.2, 0.3, 0.4]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("test_size:", test_size, "acc:", round(acc, 4))

In [ ]:
# TODO: random_state를 바꿔가며 정확도 확인
for rs in [0, 1, 42, 100, 999]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=rs
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("random_state:", rs, "acc:", round(acc, 4))

In [ ]:
from sklearn.model_selection import cross_val_score

# TODO: 교차검증으로 안정적인 성능 확인
scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("CV scores:", scores)
print("CV mean:", scores.mean(), "std:", scores.std())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: 랜덤포레스트 모델 설정
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# TODO: 전처리 + 모델 파이프라인
rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

# TODO: 교차검증 성능 출력
scores = cross_val_score(rf_clf, X, y, cv=5, scoring="accuracy")
print("RF CV mean:", scores.mean(), "std:", scores.std())

In [ ]:
# TODO: max_depth 변화에 따른 성능 비교
for depth in [2, 3, 4, 5, 7, 10, None]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=depth,
        random_state=42
    )
    rf_clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", rf)
    ])
    scores = cross_val_score(rf_clf, X, y, cv=5, scoring="accuracy")
    print("max_depth:", depth, "CV mean:", round(scores.mean(), 4))

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: 그리드서치로 최적 파라미터 찾기
rf = RandomForestClassifier(random_state=42)

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5, 7, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

gs = GridSearchCV(rf_clf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X, y)

print("Best score:", gs.best_score_)
print("Best params:", gs.best_params_)